# Model to transform the gas to an hydrogen grid

Import packages

In [72]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd

Import Data

In [85]:
# Specify the path to your Excel file
input_file_path = '../01_data/01_input_data/02_processed/'
excel_file_path = 'Data.xlsx'  

# Read the Excel file into a DataFrame
df_nodes = pd.read_excel(input_file_path + excel_file_path, sheet_name='Nodes')
df_commodities = pd.read_excel(input_file_path + excel_file_path, sheet_name='Commodities')
df_edges = pd.read_excel(input_file_path + excel_file_path, sheet_name='Edges')
df_parameter = pd.read_excel(input_file_path + excel_file_path, sheet_name='Parameters')
df_supply_values = pd.read_excel(input_file_path + excel_file_path, sheet_name='Supply')
df_demand_values = pd.read_excel(input_file_path + excel_file_path, sheet_name='Demand')

Create input data structure

In [100]:
# Extract nodes, edges and commodities from the DataFrames
Supply_nodes = df_nodes['Supply Nodes'].dropna().tolist()
Demand_nodes = df_nodes['Demand Nodes'].dropna().tolist()
Commodities = df_commodities['Commodities'].dropna().tolist()
Edges = list(zip(df_edges['Source'], df_edges['Destination']))

# Create a nested dictionary for initial capacities
Initial_capacities = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    initial_capacity = row['initial_capacities']

    edge = f"{source}{destination}"

    if commodity not in Initial_capacities:
        Initial_capacities[commodity] = {}

    Initial_capacities[commodity][edge] = initial_capacity

# Create a nested dictionary for max capacities
Max_capacities = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    max_capacity = row['max_capacities']

    edge = f"{source}{destination}"

    if commodity not in Max_capacities:
        Max_capacities[commodity] = {}

    Max_capacities[commodity][edge] = max_capacity

# Create a nested dictionary for edge cost
Edge_cost = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    edge_cost = row['costs_edge']

    edge = f"{source}{destination}"

    if commodity not in Edge_cost:
        Edge_cost[commodity] = {}

    Edge_cost[commodity][edge] = edge_cost

# Create a nested dictionary for new pipelines
Pipe_new_cost = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    new_cost = row['new_build_cost']

    edge = f"{source}{destination}"

    if commodity not in Pipe_new_cost:
        Pipe_new_cost[commodity] = {}

    Pipe_new_cost[commodity][edge] = new_cost

# Create a nested dictionary for pipeline conversion
Pipe_conv_cost = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    conv_cost = row['conversion_cost']

    edge = f"{source}{destination}"

    if commodity not in Pipe_conv_cost:
        Pipe_conv_cost[commodity] = {}

    Pipe_conv_cost[commodity][edge] = conv_cost

# Create a nested dictionary for adjusting the capacity when pipeline conversion
Pipe_conv_factor = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    conv_cap_factor = row['conversion_capacity_factor']

    edge = f"{source}{destination}"

    if commodity not in Pipe_conv_factor:
        Pipe_conv_factor[commodity] = {}

    Pipe_conv_factor[commodity][edge] = conv_cap_factor

# Create a nested dictionary for supply values, skipping 0 and NaN values
Supply_values = {}
for index, row in df_supply_values.iterrows():
    commodity = row['Commodity']
    supply_node = row['Node']
    supply_value = row['Supply']

    if commodity not in Supply_values:
        Supply_values[commodity] = {}

    # Skip 0 and NaN values
    if not pd.isna(supply_value) and supply_value != 0:
        Supply_values[commodity][supply_node] = supply_value

# Create a nested dictionary for demand values, skipping 0 and NaN values
Demand_values = {}
for index, row in df_demand_values.iterrows():
    commodity = row['Commodity']
    demand_node = row['Node']
    demand_value = row['Demand']

    if commodity not in Demand_values:
        Demand_values[commodity] = {}

    # Skip 0 and NaN values
    if not pd.isna(demand_value) and demand_value != 0:
        Demand_values[commodity][demand_node] = demand_value

Print data structure for control

In [101]:
# Print the data
print("Supply Nodes:", Supply_nodes)
print("Demand Nodes:", Demand_nodes)
print("Commodities:", Commodities)
print("Edges:", Edges)
print("Initial Capacities:", Initial_capacities)
print("Max Capacities:", Max_capacities)
print("Costs per edge Capacities:", Edge_cost)
print("Costs for new pipelines:", Pipe_new_cost)
print("Costs for convert pipelines:", Pipe_conv_cost)
print("Conversion capacity factor:", Pipe_conv_factor)
print("Supply Values:", Supply_values)
print("Demand Values:", Demand_values)

Supply Nodes: ['S1', 'S2', 'S3']
Demand Nodes: ['D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10']
Commodities: ['Methane', 'Hydrogen']
Edges: [('S1', 'D1'), ('S1', 'D2'), ('S2', 'D2'), ('S2', 'D3'), ('S3', 'D1'), ('D1', 'D2'), ('D3', 'D4'), ('D4', 'D5'), ('D4', 'D6'), ('D6', 'D7'), ('D5', 'D8'), ('D5', 'D9'), ('D9', 'D10'), ('D7', 'D10')]
Initial Capacities: {'Methane': {'S1D1': 20, 'S1D2': 30, 'S2D2': 25, 'S2D3': 40, 'S3D1': 35, 'D1D2': 30, 'D3D4': 30, 'D4D5': 40, 'D4D6': 40, 'D6D7': 30, 'D5D8': 15, 'D5D9': 10, 'D9D10': 10, 'D7D10': 20}, 'Hydrogen': {'S1D1': 0, 'S1D2': 0, 'S2D2': 0, 'S2D3': 0, 'S3D1': 0, 'D1D2': 0, 'D3D4': 0, 'D4D5': 0, 'D4D6': 0, 'D6D7': 0, 'D5D8': 0, 'D5D9': 0, 'D9D10': 0, 'D7D10': 0}}
Max Capacities: {'Methane': {'S1D1': 25, 'S1D2': 40, 'S2D2': 35, 'S2D3': 50, 'S3D1': 45, 'D1D2': 30, 'D3D4': 30, 'D4D5': 40, 'D4D6': 40, 'D6D7': 30, 'D5D8': 15, 'D5D9': 10, 'D9D10': 10, 'D7D10': 20}, 'Hydrogen': {'S1D1': 50, 'S1D2': 80, 'S2D2': 70, 'S2D3': 100, 'S3D1': 90, 

In [107]:
#implement factor to adjust capacity when conversion from methane to hydrogen
#TODO Implement it from the input file and use a correct factor
conversion_factor = Pipe_conv_factor
conversion_factor

{'Methane': {'S1D1': 1,
  'S1D2': 1,
  'S2D2': 1,
  'S2D3': 1,
  'S3D1': 1,
  'D1D2': 1,
  'D3D4': 1,
  'D4D5': 1,
  'D4D6': 1,
  'D6D7': 1,
  'D5D8': 1,
  'D5D9': 1,
  'D9D10': 1,
  'D7D10': 1},
 'Hydrogen': {'S1D1': 2,
  'S1D2': 2,
  'S2D2': 2,
  'S2D3': 2,
  'S3D1': 2,
  'D1D2': 2,
  'D3D4': 2,
  'D4D5': 2,
  'D4D6': 2,
  'D6D7': 2,
  'D5D8': 2,
  'D5D9': 2,
  'D9D10': 2,
  'D7D10': 2}}

Create model

In [103]:
# Create a new model
model = gp.Model("Grid_Transformation")

Define parameters

In [104]:
# Parameters
supply_nodes = Supply_nodes  # Supply nodes
demand_nodes = Demand_nodes  # Demand nodes
commodities = Commodities  # Commodity types
edges = Edges  # Edges
initial_capacities = Initial_capacities # Initial capacities
max_capacities = Max_capacities  # Maximum capacities
costs_edge = Edge_cost  # Cost to transport from node to node
capacity_new_cost = Pipe_new_cost  # Cost to increase capacity
capacity_change_cost = Pipe_conv_cost  # Cost to increase capacity

supply_values = Supply_values  # Supply values
demand_values = Demand_values  # Demand values

Define decision variables

In [109]:
# Decision variables
x_flow = {} #flow of commodity on an edge
y_new_cap = {} #new build capacity for a commodity on an edge between two edges
z_conv_cap = {} #capacity of a commodity converted on an edge between two nodes
Change = {} # Binary variable for switching

for commodity in commodities:
    x_flow[commodity] = {}
    y_new_cap[commodity] = {}
    z_conv_cap[commodity] = {}
    Change[commodity] = {}
    for edge in edges:
        x_flow[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"x_{commodity}_{edge[0]}_{edge[1]}")
        y_new_cap[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"y_{commodity}_{edge[0]}_{edge[1]}")
        z_conv_cap[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"z_{commodity}_{edge[0]}_{edge[1]}")
        Change[commodity][edge] = model.addVar(vtype=GRB.BINARY, name=f"w_{commodity}_{edge[0]}_{edge[1]}")  

Define objective and constraints

In [110]:
# Objective function (minimize total transportation cost + cost to increase capacity)
model.setObjective(
    gp.quicksum(x_flow[commodity][edge] * costs_edge[commodity][f"{edge[0]}{edge[1]}"] for commodity in commodities for edge in edges) +
    gp.quicksum(y_new_cap[commodity][edge] * capacity_new_cost[commodity][f"{edge[0]}{edge[1]}"] for commodity in commodities for edge in edges) +
    gp.quicksum(z_conv_cap[commodity][edge] * capacity_change_cost[commodity][f"{edge[0]}{edge[1]}"] for commodity in commodities for edge in edges),
    GRB.MINIMIZE
)

# Constraints

# Supply constraints
for node in supply_nodes:
    for commodity in commodities:
        model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in edges if edge[0] == node) 
                        <= supply_values[commodity][node], f"supply_{commodity}_{node}")

# Demand constraints
for node in demand_nodes:
    for commodity in commodities:
        model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in edges if edge[1] == node) 
                        == demand_values[commodity][node], f"demand_{commodity}_{node}")

#Capacity constraint for flow
for commodity in commodities:
    for edge in edges:
        model.addConstr(x_flow[commodity][edge] 
                        <= y_new_cap[commodity][edge] + z_conv_cap[commodity][edge] #+ initial_capacities[commodity][f"{edge[0]}{edge[1]}"]
                        , f"used_capacity_{commodity}_{edge[0]}_{edge[1]}")

# Capacity constraint for maximal capacity
for commodity in commodities:
    for edge in edges:
        model.addConstr(y_new_cap[commodity][edge] + z_conv_cap[commodity][edge] 
                        <= max_capacities[commodity][f"{edge[0]}{edge[1]}"], f"total_capacity_{commodity}_{edge[0]}_{edge[1]}")

#Constraint for switch
model.addConstr(Change[Commodities[0]] + Change[Commodities[1]] == 1, "switching_constraint")
for commodity in commodities:
    for edge in edges:
        model.addConstr(initial_capacities[Commodities[0]][f"{edge[0]}{edge[1]}"] * Change[commodity] * conversion_factor[commodity][node]
                        == z_conv_cap[commodity][edge], f"changed_capacity_{commodity}_{edge[0]}_{edge[1]}")

#Constraing no negative flow
for commodity in commodities:
    for edge in edges:
        model.addConstr(x_flow[commodity][edge] >= 0, f"non_negativity_x_{commodity}_{edge[0]}_{edge[1]}")

TypeError: unsupported operand type(s) for +: 'dict' and 'dict'

Optimize the model

In [95]:
# Optimize the model
model.optimize()

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11.0 (22621.2))

CPU model: Intel(R) Core(TM) i5-8265U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 139 rows, 86 columns and 264 nonzeros
Model fingerprint: 0x5dfdef08
Variable types: 84 continuous, 2 integer (2 binary)
Coefficient statistics:
  Matrix range     [1e+00, 8e+01]
  Objective range  [5e-01, 1e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+02]
Presolve removed 48 rows and 39 columns
Presolve time: 0.00s

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 0

Model is infeasible
Best objective -, best bound -, gap -


Results processing

In [96]:
# Print the results
if model.status == GRB.OPTIMAL:
    print("Optimal solution found!")
    for commodity in commodities:
        for edge in edges:
            print(f"{commodity}, {edge}: Flow of Commodity = {x_flow[commodity][edge].x}, New Capacity = {y_new_cap[commodity][edge].x}, Switched = {Change[commodity].x}, Changed Capacity = {z_conv_cap[commodity][edge].x}")
    print("****************************")
    print(f"Total cost: {model.objVal}")
else:
    print("No optimal solution found.")

No optimal solution found.
